In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [8]:
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/behavior-aware-bms"

df = pd.read_csv(
    f"{PROJECT_ROOT}/data/processed/nasa/nasa_analysis_features_v2.csv"
)

print(df.shape)
print(df.columns.tolist())

(7282946, 21)
['battery_id', 'cycle', 'timestamp', 'voltage_v', 'current_a', 'temperature_c', 'capacity_ah', 'soc', 'soh', 'power_w', 'mode_guess', 'source', 'source_file', 'aggressive_discharge_event', 'fast_charge_duration', 'deep_discharge_duration', 'high_temp_duration', 'deep_discharge_frequency', 'high_soc_duration', 'raw_stress_score_v2', 'stress_score_v2']


In [10]:
df = df.sort_values(
    ["battery_id", "cycle"]
).reset_index(drop=True)

In [11]:
df["stress_rolling_mean"] = (
    df.groupby("battery_id")["stress_score_v2"]
      .transform(
          lambda x: x.rolling(
              window=50,
              min_periods=1
          ).mean()
      )
)

In [12]:
df["stress_rolling_std"] = (
    df.groupby("battery_id")["stress_score_v2"]
      .transform(
          lambda x: x.rolling(
              window=50,
              min_periods=1
          ).std()
      )
)

df["stress_rolling_std"] = (
    df["stress_rolling_std"]
    .fillna(0)
)

In [13]:
df["temp_rolling_mean"] = (
    df.groupby("battery_id")["temperature_c"]
      .transform(
          lambda x: x.rolling(
              window=50,
              min_periods=1
          ).mean()
      )
)

df["temp_rolling_max"] = (
    df.groupby("battery_id")["temperature_c"]
      .transform(
          lambda x: x.rolling(
              window=50,
              min_periods=1
          ).max()
      )
)

In [14]:
max_cycle = df["cycle"].max()

df["battery_age_factor"] = (
    df["cycle"] / max_cycle
)

In [15]:
df["cycle_stress_index"] = (
    df["stress_score_v2"]
    *
    df["battery_age_factor"]
)

In [16]:
new_cols = [
    "stress_rolling_mean",
    "stress_rolling_std",
    "temp_rolling_mean",
    "temp_rolling_max",
    "battery_age_factor",
    "cycle_stress_index"
]

df[new_cols].describe()

,stress_rolling_mean,stress_rolling_std,temp_rolling_mean,temp_rolling_max,battery_age_factor,cycle_stress_index
count,7.282946e+06,7.282946e+06,7.282946e+06,7.282946e+06,7.282946e+06,7.282946e+06
mean,8.246183e+00,2.097774e-01,2.220873e+01,2.262302e+01,2.659126e-01,1.862199e+00
std,9.259002e+00,9.216447e-01,1.235562e+01,1.245688e+01,2.497570e-01,2.550252e+00
min,0.000000e+00,0.000000e+00,3.547502e+00,4.161812e+00,0.000000e+00,0.000000e+00
25%,1.900000e-01,0.000000e+00,7.378763e+00,7.859449e+00,6.991870e-02,1.235772e-02
50%,5.220000e+00,0.000000e+00,2.469077e+01,2.476513e+01,1.804878e-01,5.107317e-01
75%,1.375000e+01,0.000000e+00,2.759368e+01,2.785319e+01,3.918699e-01,3.104000e+00
max,4.484000e+01,1.718733e+01,6.649517e+01,6.986975e+01,1.000000e+00,1.887746e+01


In [17]:
battery_summary = (
    df.groupby("battery_id")
      .agg({
          "stress_score_v2":"mean",
          "temperature_c":"mean",
          "fast_charge_duration":"sum",
          "deep_discharge_duration":"sum",
          "high_temp_duration":"sum",
          "aggressive_discharge_event":"sum",
          "soc":"mean"
      })
      .reset_index()
)

battery_summary.columns = [
    "battery_id",
    "avg_stress",
    "avg_temp",
    "fast_charge_duration",
    "deep_discharge_duration",
    "high_temp_duration",
    "aggressive_discharge_count",
    "avg_soc"
]

battery_summary.head()

,battery_id,avg_stress,avg_temp,fast_charge_duration,deep_discharge_duration,high_temp_duration,aggressive_discharge_count,avg_soc
0,B0005,13.670454,26.369701,1.250087e+09,1.277054e+07,3581775.549,45284,NaN
1,B0006,11.407560,26.429154,1.027942e+09,1.931800e+07,4680506.022,44512,NaN
2,B0007,0.065179,26.119363,6.213354e+05,1.428340e+07,4420739.468,195,NaN
3,B0018,14.449429,25.913199,6.962533e+08,1.054231e+07,0.000,32084,NaN
4,B0025,16.337941,28.482356,2.693701e+08,1.155482e+07,2827773.483,4349,NaN


In [18]:
Path(
    f"{PROJECT_ROOT}/data/features"
).mkdir(
    parents=True,
    exist_ok=True
)
df.to_csv(
    f"{PROJECT_ROOT}/data/features/behavior_features_v3.csv",
    index=False
)
battery_summary.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv",
    index=False
)
print("T16 files saved.")

T16 files saved.


In [19]:
import os

print(os.path.exists(
    f"{PROJECT_ROOT}/data/features/behavior_features_v3.csv"
))

print(os.path.exists(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv"
))

True
True


In [21]:
battery_summary.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv",
    index=False
)

In [22]:
feature_catalog = pd.DataFrame({
    "feature":[
        "stress_rolling_mean",
        "stress_rolling_std",
        "temp_rolling_mean",
        "temp_rolling_max",
        "battery_age_factor",
        "cycle_stress_index"
    ]
})

feature_catalog.to_csv(
    f"{PROJECT_ROOT}/reports/metrics/t16_feature_catalog.csv",
    index=False
)

In [23]:
sample_df = df.sample(
    n=100000,
    random_state=42
)

sample_df.to_csv(
    f"{PROJECT_ROOT}/data/features/behavior_features_v3_sample.csv",
    index=False
)

In [24]:
battery_summary.head()

battery_summary.shape

(34, 8)

In [25]:
from pathlib import Path

Path(
    f"{PROJECT_ROOT}/reports/metrics"
).mkdir(
    parents=True,
    exist_ok=True
)

feature_catalog = pd.DataFrame({
    "feature":[
        "stress_rolling_mean",
        "stress_rolling_std",
        "temp_rolling_mean",
        "temp_rolling_max",
        "battery_age_factor",
        "cycle_stress_index"
    ]
})

feature_catalog.to_csv(
    f"{PROJECT_ROOT}/reports/metrics/t16_feature_catalog.csv",
    index=False
)

print("saved")

saved


In [26]:
battery_summary.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv",
    index=False
)

print("battery summary saved")

battery summary saved


In [27]:
import os

files = [
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv",
    f"{PROJECT_ROOT}/reports/metrics/t16_feature_catalog.csv"
]

for f in files:
    print(os.path.exists(f), f)

True /content/drive/MyDrive/behavior-aware-bms/data/features/battery_summary_v1.csv
True /content/drive/MyDrive/behavior-aware-bms/reports/metrics/t16_feature_catalog.csv


In [29]:
%cd /content/drive/MyDrive/behavior-aware-bms

/content/drive/MyDrive/behavior-aware-bms


In [30]:
!pwd

/content/drive/MyDrive/behavior-aware-bms


In [31]:
!git status

Refresh index: 100% (66/66), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/features/
	data/raw/nasa/cleaned_dataset/
	notebooks/Untitled0.ipynb
	notebooks/nasa_preproc
	reports/metrics/t16_feature_catalog.csv


It took 2.54 seconds to enumerate untracked files. 'status -uno'
may speed it up, but you have to be careful not to forget to add
new files yourself (see 'git help status').
no changes added to commit (use "git add" and/or "git commit -a")


In [32]:
!ls -lh /content/drive/MyDrive/behavior-aware-bms/data/features



total 1.8G
-rw------- 1 root root 2.9K Jun 25 14:10 battery_summary_v1.csv
-rw------- 1 root root 1.8G Jun 25 14:01 behavior_features_v3.csv
-rw------- 1 root root  25M Jun 25 14:09 behavior_features_v3_sample.csv


In [33]:
!ls -lh /content/drive/MyDrive/behavior-aware-bms/notebooks


total 1.2M
-rw------- 1 root root  389 Jun 22 04:00 03_calce_preprocessing.ipynb
-rw------- 1 root root 991K Jun 24 02:55 BMS_Project_Setup.ipynb
-rw------- 1 root root 143K Jun 24 08:23 error_analysis.ipynb
-rw------- 1 root root  20K Jun 24 07:43 nasa_preproc
-rw------- 1 root root  35K Jun 25 14:15 Untitled0.ipynb


In [35]:
%cd /content/drive/MyDrive/behavior-aware-bms

!git add notebooks/04_feature_refinement.ipynb

!git add data/features/battery_summary_v1.csv

!git add reports/metrics/t16_feature_catalog.csv

/content/drive/MyDrive/behavior-aware-bms


In [36]:
!git status

On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/features/battery_summary_v1.csv
	new file:   notebooks/04_feature_refinement.ipynb
	new file:   reports/metrics/t16_feature_catalog.csv

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/04_feature_refinement.ipynb
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/features/behavior_features_v3.csv
	data/features/behavior_features_v3_sample.csv
	data/raw/nasa/cleaned_dataset/
	notebooks/nasa_preproc

